# 01 Data Exploration: PMP Dataset Overview

This notebook explores the peripheral membrane protein (PMP) dataset:
- Dataset statistics (size, binding residue distribution)
- Structure quality metrics
- Feature distributions
- Class imbalance analysis
- Protein family breakdown

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Dataset

In [ ]:
# Load split files
splits_dir = Path('data/splits')
train_ids = Path('data/splits/train.txt').read_text().strip().split('\n')
val_ids = Path('data/splits/val.txt').read_text().strip().split('\n')
test_ids = Path('data/splits/test.txt').read_text().strip().split('\n')

print(f'Train proteins: {len(train_ids)}')
print(f'Val proteins: {len(val_ids)}')
print(f'Test proteins: {len(test_ids)}')
print(f'Total: {len(train_ids) + len(val_ids) + len(test_ids)}')

## 2. Binding Residue Statistics

In [ ]:
# Load binding labels
labels_dir = Path('data/raw/labels')

binding_stats = []
all_binding_counts = []
all_non_binding_counts = []

for pdb_id in train_ids[:10]:  # sample first 10 for demo
    label_file = labels_dir / f'{pdb_id}_binding.csv'
    if label_file.exists():
        df = pd.read_csv(label_file)
        n_binding = (df['is_binding'] == 1).sum()
        n_total = len(df)
        n_non_binding = n_total - n_binding
        
        binding_stats.append({
            'pdb_id': pdb_id,
            'total_residues': n_total,
            'binding_residues': n_binding,
            'non_binding_residues': n_non_binding,
            'binding_fraction': n_binding / n_total
        })
        
        all_binding_counts.append(n_binding)
        all_non_binding_counts.append(n_non_binding)

stats_df = pd.DataFrame(binding_stats)
print(stats_df)
print(f'\nAverage binding fraction: {stats_df["binding_fraction"].mean():.2%}')
print(f'Std binding fraction: {stats_df["binding_fraction"].std():.2%}')

## 3. Class Imbalance Visualization

In [ ]:
# Binding vs non-binding distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
total_binding = sum(all_binding_counts)
total_non_binding = sum(all_non_binding_counts)

axes[0].bar(['Binding', 'Non-binding'], [total_binding, total_non_binding], color=['red', 'blue'])
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution')
axes[0].set_yscale('log')

# Pie chart
axes[1].pie(
    [total_binding, total_non_binding],
    labels=['Binding', 'Non-binding'],
    colors=['red', 'blue'],
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Class Imbalance')

plt.tight_layout()
plt.savefig('outputs/data_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Binding residues: {total_binding} ({total_binding/(total_binding+total_non_binding):.1%})')
print(f'Non-binding residues: {total_non_binding} ({total_non_binding/(total_binding+total_non_binding):.1%})')

## 4. Protein Length Distribution

In [ ]:
# Protein lengths
lengths = stats_df['total_residues'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(lengths, bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Protein Length (residues)')
axes[0].set_ylabel('Count')
axes[0].set_title('Protein Length Distribution')
axes[0].axvline(np.mean(lengths), color='r', linestyle='--', label=f'Mean: {np.mean(lengths):.0f}')
axes[0].legend()

# Box plot by split
split_data = {
    'Train': stats_df['total_residues'].values,
    'Val': [300, 350, 280],  # placeholder
    'Test': [320, 370, 290]   # placeholder
}

axes[1].boxplot([split_data['Train'], split_data['Val'], split_data['Test']], 
                labels=['Train', 'Val', 'Test'])
axes[1].set_ylabel('Protein Length (residues)')
axes[1].set_title('Length Distribution by Split')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/data_protein_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean length: {np.mean(lengths):.1f} residues')
print(f'Std length: {np.std(lengths):.1f} residues')
print(f'Min length: {np.min(lengths)} residues')
print(f'Max length: {np.max(lengths)} residues')

## 5. Binding Residue Density

In [ ]:
# How are binding residues distributed along sequence?
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Binding fraction histogram
axes[0, 0].hist(stats_df['binding_fraction'] * 100, bins=15, edgecolor='black')
axes[0, 0].set_xlabel('Binding Residue Fraction (%)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Distribution of Binding Residue Fraction')
axes[0, 0].axvline(stats_df['binding_fraction'].mean() * 100, color='r', linestyle='--')

# Absolute binding count
axes[0, 1].hist(stats_df['binding_residues'], bins=15, edgecolor='black')
axes[0, 1].set_xlabel('Number of Binding Residues')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Distribution of Binding Residue Counts')

# Scatter: total vs binding
axes[1, 0].scatter(stats_df['total_residues'], stats_df['binding_residues'], s=100, alpha=0.6)
axes[1, 0].set_xlabel('Total Residues')
axes[1, 0].set_ylabel('Binding Residues')
axes[1, 0].set_title('Binding Count vs Total Length')

# Correlation
correlation = stats_df['total_residues'].corr(stats_df['binding_residues'])
axes[1, 1].text(0.5, 0.5, f'Correlation: {correlation:.3f}', 
                ha='center', va='center', fontsize=16, transform=axes[1, 1].transAxes)
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('outputs/data_binding_density.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary Statistics

In [ ]:
print('="'*50)
print('DATASET SUMMARY')
print('="'*50)
print(f'\nTotal proteins: {len(train_ids) + len(val_ids) + len(test_ids)}')
print(f'Train/Val/Test split: {len(train_ids)}/{len(val_ids)}/{len(test_ids)}')
print(f'\nAverage protein length: {stats_df["total_residues"].mean():.1f} ± {stats_df["total_residues"].std():.1f}')
print(f'Total residues: {stats_df["total_residues"].sum()}')
print(f'\nBinding residues: {total_binding} ({total_binding/(total_binding+total_non_binding)*100:.1f}%)')
print(f'Non-binding residues: {total_non_binding} ({total_non_binding/(total_binding+total_non_binding)*100:.1f}%)')
print(f'\nClass imbalance ratio: {total_non_binding/total_binding:.1f}:1')
print(f'Average binding fraction per protein: {stats_df["binding_fraction"].mean():.1%}')
print('\n' + '="'*50)